## schema evolution with auto loader

In [0]:
#auto loader, reading the csv file
df=spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","csv") \
    .option("cloudFiles.schemaLocation","/Volumes/workspace/default/autoloader_sink/schema") \
    .option("schemaEvolutionMode","addNewColumns") \
    .load("/Volumes/workspace/default/autoloader_source/source")   

   

In [0]:
df.writeStream.format("delta") \
    .option("checkpointLocation","/Volumes/workspace/default/autoloader_sink/checkpoint") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("workspace.default.autoloader_sink")

   




In [0]:
%sql
select * from autoloader_sink

##  schema evolution without auto loader

In [0]:
data1=[(1,"India","Delhi"),
       (2,"India","Mumbai"),
       (3,"India","Bangalore"),
       (4,"India","Chennai"),
       (5,"India","Kolkata")
       ]
data2=[(1,"india","delhi","new delhi"),
       (2,"india","mumbai","maharastra"),
       (3,"india","bangalore","karnataka"),
       (4,"india","chennai","tamilnadu"),
       (5,"india","kolkata","west bengal")]

data3=[(1,"India","Delhi"),
       (2,"India","Mumbai"),
       (3,"India","Bangalore"),
       (4,"India","Chennai"),
       (5,"India","Kolkata")
       ]

df1=spark.createDataFrame(data1,["customer_id","country","city"])
df2=spark.createDataFrame(data2,["customer_id","country","city","state"])
df3=spark.createDataFrame(data3,["customer_id","country","city"])

display(df1)
display(df2)
display(df3)

In [0]:
# write df1 into table
df1.write.format("delta").mode("append").saveAsTable("workspace.default.df1")


In [0]:
%sql
select * from df1

In [0]:
#load 2nd data frame into df1 using merge schema option
df2.write.format("delta").option("mergeSchema","true").mode("append").saveAsTable("workspace.default.df1")

In [0]:
%sql
select * from df1

In [0]:
#load 3rd data frame into df1 using merge schema option
df3.write.format("delta").option("mergeSchema","true").mode("append").saveAsTable("workspace.default.df1")

In [0]:
%sql
select * from df1

## Flatten list

In [0]:
from pyspark.sql.functions import explode,split,col,trim
input=[("ravi","cricket,badminton"),
       ("ram","football,cricket"),
       ("shyam","hockey,cricket"),
       ("rajesh","cricket,football"),
       ("suresh","cricket,hockey")]
df=spark.createDataFrame(input,["name","sports"])   
df.display()

#split sports into rows and display
df.select("name",(explode(split(trim("sports"),",")))).orderBy("name").display()

